# 09 Text Feature Extraction

Extract missing structured features (`num_floors`, `bedrooms`, `bathrooms`, `frontage_m`, `road_width_m`, `width_m`, `length_m`) from listing `title` + `description` text using regex patterns common in Vietnamese real estate listings.

Shows before/after comparison of missingness rates across the full dataset.

In [ ]:
from pathlib import Path
import sys

SCENE_ROOT = Path('/Users/dienmayhaituyet/Documents/AVM/Rule-based Scene')
SRC_DIR = SCENE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from common import (
    load_raw_data, preprocess_raw_data, clean_measurement, RAW_DATA_DIR, EVAL_DIR
)
from extract_from_text import extract_features_from_text, TARGET_COLS

## 1. Baseline missingness (structured columns only, no text extraction)

In [ ]:
print('Loading raw data...')
df_raw = pd.read_csv(RAW_DATA_DIR / 'mick_df_full_raw.csv', nrows=250_000)
print(f'Raw rows: {len(df_raw):,}')

# Reproduce only the numeric_map step to get a clean baseline
COLS_MAP = {
    'number_of_floors': ('num_floors',    0,   100),
    'house_front':      ('frontage_m',    0.5, 200),
    'road_wide':        ('road_width_m',  0.5, 200),
    'bedroom':          ('bedrooms',      0,   100),
    'bathroom':         ('bathrooms',     0,   100),
    'width':            ('width_m',       0.5, 200),
    'length':           ('length_m',      0.5, 1000),
}

df_base = df_raw.copy()
for raw_col, (new_col, lo, hi) in COLS_MAP.items():
    df_base[new_col] = df_base[raw_col].apply(
        lambda x: clean_measurement(x, min_val=lo, max_val=hi)
    )

N = len(df_base)
baseline_missing = {new_col: int(df_base[new_col].isna().sum())
                    for _, (new_col, _, _) in COLS_MAP.items()}

print(f'\nBaseline missingness (N={N:,}):')
for col, cnt in baseline_missing.items():
    print(f'  {col:15s}: {cnt:6,d} missing  ({cnt/N*100:.1f}%)')

## 2. After text extraction

In [ ]:
print('Running full preprocess (includes text extraction)...')
df_after = preprocess_raw_data(df_raw)
N2 = len(df_after)
print(f'Rows after dedup: {N2:,}')

after_missing = {col: int(df_after[col].isna().sum()) for col in TARGET_COLS}

print(f'\nAfter-extraction missingness (N={N2:,}):')
for col in TARGET_COLS:
    b = baseline_missing[col]
    a = after_missing[col]
    filled = b - a
    recovery = filled / b * 100 if b > 0 else 0.0
    print(f'  {col:15s}: {a:6,d} missing ({a/N2*100:.1f}%)  '
          f'| filled {filled:6,d}  ({recovery:.1f}% of previously missing)')

## 3. Precision check — agreement with existing structured values

In [ ]:
# For rows where the structured column already had a value AND text also has a signal,
# check how often they agree (within ±10% for continuous, exact for integer features).

from common import normalize_text
from extract_from_text import (
    _extract_num_floors, _extract_bedrooms, _extract_bathrooms,
    _extract_frontage, _extract_road_width, _extract_width_length
)

df_base['all_text_lower'] = (
    df_base['title'].fillna('') + ' ' + df_base['description'].fillna('')
).map(normalize_text)

extractors = {
    'num_floors':   _extract_num_floors,
    'bedrooms':     _extract_bedrooms,
    'bathrooms':    _extract_bathrooms,
    'frontage_m':   _extract_frontage,
    'road_width_m': _extract_road_width,
}

print('Precision vs existing structured values (rows where both exist):')
print(f'  {"Feature":15s}  {"N checked":>10s}  {"Exact match":>12s}  {"Within 10%":>12s}')
for col, fn in extractors.items():
    has_val = df_base[col].notna()
    if not has_val.any():
        continue
    extracted = df_base.loc[has_val, 'all_text_lower'].map(fn)
    both = extracted.notna()
    if not both.any():
        continue
    actual   = df_base.loc[has_val, col][both]
    pred     = extracted[both]
    exact    = (actual == pred).mean()
    within10 = (((actual - pred).abs() / actual.abs().clip(lower=1e-6)) <= 0.10).mean()
    print(f'  {col:15s}  {both.sum():>10,d}  {exact*100:>11.1f}%  {within10*100:>11.1f}%')

# width/length need special handling
for col, idx in [('width_m', 0), ('length_m', 1)]:
    has_val = df_base[col].notna()
    if not has_val.any():
        continue
    extracted = df_base.loc[has_val, 'all_text_lower'].map(
        lambda t: _extract_width_length(t)[idx]
    )
    both = extracted.notna()
    if not both.any():
        continue
    actual   = df_base.loc[has_val, col][both]
    pred     = extracted[both]
    exact    = (actual == pred).mean()
    within10 = (((actual - pred).abs() / actual.abs().clip(lower=1e-6)) <= 0.10).mean()
    print(f'  {col:15s}  {both.sum():>10,d}  {exact*100:>11.1f}%  {within10*100:>11.1f}%')

## 4. Before / After visualisation

In [ ]:
LABEL_MAP = {
    'num_floors':   'num_floors',
    'bedrooms':     'bedrooms',
    'bathrooms':    'bathrooms',
    'frontage_m':   'frontage_m',
    'road_width_m': 'road_width_m',
    'width_m':      'width_m',
    'length_m':     'length_m',
}
labels     = list(LABEL_MAP.values())
before_pct = [baseline_missing[c] / N * 100 for c in TARGET_COLS]
after_pct  = [after_missing[c]    / N2 * 100 for c in TARGET_COLS]
filled_pct = [
    (baseline_missing[c] - after_missing[c]) / N * 100
    for c in TARGET_COLS
]

x = range(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Text Extraction: Before vs After Missingness', fontsize=14, fontweight='bold')

# --- Left: grouped bar (before / after) ---
ax = axes[0]
bars_b = ax.bar([i - width/2 for i in x], before_pct, width, label='Before', color='#d9534f', alpha=0.85)
bars_a = ax.bar([i + width/2 for i in x], after_pct,  width, label='After',  color='#5cb85c', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Missing %')
ax.set_title('Missing % before and after text extraction')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
for bar in bars_a:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

# --- Right: rows filled (absolute) ---
ax2 = axes[1]
filled_abs = [baseline_missing[c] - after_missing[c] for c in TARGET_COLS]
recovery   = [
    (baseline_missing[c] - after_missing[c]) / baseline_missing[c] * 100
    if baseline_missing[c] > 0 else 0
    for c in TARGET_COLS
]
colors = ['#337ab7' if v > 0 else '#aaa' for v in filled_abs]
bars_f = ax2.bar(list(x), filled_abs, color=colors, alpha=0.85)
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=35, ha='right', fontsize=10)
ax2.set_ylabel('Rows filled')
ax2.set_title('Rows filled by text extraction (with recovery %)')
for bar, rec in zip(bars_f, recovery):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{rec:.1f}%', ha='center', va='bottom', fontsize=9, color='#333')

plt.tight_layout()
out_path = SCENE_ROOT / 'Visualizations' / 'visualizations' / 'text_extraction_before_after.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {out_path}')

## 5. Value distribution of extracted values

In [ ]:
# Show distribution of extracted (newly-filled) values to spot implausible outliers
from common import normalize_text
from extract_from_text import (
    _extract_num_floors, _extract_bedrooms, _extract_bathrooms,
    _extract_frontage, _extract_road_width, _extract_width_length
)

df_base['all_text_lower'] = (
    df_base['title'].fillna('') + ' ' + df_base['description'].fillna('')
).map(normalize_text)

was_missing = {col: df_base[col].isna() for col in TARGET_COLS}

extractors_scalar = {
    'num_floors':   _extract_num_floors,
    'bedrooms':     _extract_bedrooms,
    'bathrooms':    _extract_bathrooms,
    'frontage_m':   _extract_frontage,
    'road_width_m': _extract_road_width,
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Distribution of extracted values (rows that were missing)', fontsize=13)
axes = axes.flatten()

plot_idx = 0
for col, fn in extractors_scalar.items():
    mask = was_missing[col]
    extracted = df_base.loc[mask, 'all_text_lower'].map(fn).dropna()
    ax = axes[plot_idx]
    ax.hist(extracted, bins=30, color='#337ab7', alpha=0.8, edgecolor='white')
    ax.set_title(f'{col}\n(n={len(extracted):,} filled)', fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    plot_idx += 1

for col, idx in [('width_m', 0), ('length_m', 1)]:
    mask = was_missing[col]
    extracted = df_base.loc[mask, 'all_text_lower'].map(
        lambda t: _extract_width_length(t)[idx]
    ).dropna()
    ax = axes[plot_idx]
    ax.hist(extracted, bins=30, color='#5cb85c', alpha=0.8, edgecolor='white')
    ax.set_title(f'{col}\n(n={len(extracted):,} filled)', fontsize=10)
    ax.set_xlabel('Value (m)')
    ax.set_ylabel('Count')
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
out_path2 = SCENE_ROOT / 'Visualizations' / 'visualizations' / 'text_extraction_value_distributions.png'
plt.savefig(out_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {out_path2}')

## 6. Summary table

In [ ]:
rows = []
for col in TARGET_COLS:
    b = baseline_missing[col]
    a = after_missing[col]
    filled = b - a
    rows.append({
        'feature':          col,
        'missing_before':   b,
        'missing_pct_before': round(b / N * 100, 1),
        'missing_after':    a,
        'missing_pct_after': round(a / N2 * 100, 1),
        'rows_filled':      filled,
        'recovery_pct':     round(filled / b * 100, 1) if b > 0 else 0.0,
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

out_csv = SCENE_ROOT / 'Evaluation' / 'improvements' / 'text_extraction_summary.csv'
summary_df.to_csv(out_csv, index=False)
print(f'\nSaved to {out_csv}')